# 01 — Exploratory Data Analysis
**Global Market Intelligence | Yahoo Finance RapidAPI**

This notebook covers:
1. Data loading & shape inspection
2. Descriptive statistics
3. Missing value analysis
4. Return distributions & fat-tail tests
5. Sector-level comparisons
6. Correlation analysis
7. Volatility clustering (ARCH effects)
8. Outlier visualisation
9. Technical indicator deep-dive

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5),
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('tab10')

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
RAW_DIR  = ROOT / 'data' / 'raw'
print('Paths OK — ROOT:', ROOT)

## 1  Load Data

In [ ]:
# Run pipeline first if processed data does not exist
clean_path = PROC_DIR / 'cy_clean.parquet'
if not clean_path.exists():
    print('Running ingestion + cleaning pipeline...')
    from src.ingestion import run_ingestion
    from src.cleaning  import run_cleaning
    run_ingestion()
    run_cleaning()

df = pd.read_parquet(clean_path)
df['date'] = pd.to_datetime(df['date'])
print(f'Shape: {df.shape}')
df.head(3)

## 2  Descriptive Statistics

In [ ]:
print('=== Dataset Overview ===')
print(f'Tickers : {df["ticker"].nunique()}')
print(f'Sectors : {df["sector"].nunique()}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Total rows : {len(df):,}')
print()

num_cols = ['open','high','low','close','volume','daily_return','vol_20d']
df[num_cols].describe().round(4)

## 3  Missing Value Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
null_pct = df.isnull().mean().mul(100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
if len(null_pct) > 0:
    ax.bar(null_pct.index, null_pct.values, color='#cc0000')
    ax.set_title('Missing Value Rate (%)')
    ax.set_ylabel('%')
    plt.xticks(rotation=45, ha='right')
else:
    ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=14)
    ax.set_title('Missing Values')
plt.tight_layout()
plt.show()

## 4  Return Distribution & Normality Tests

In [ ]:
rets = df['daily_return'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram with normal overlay
mu, sigma = rets.mean(), rets.std()
axes[0].hist(rets, bins=150, density=True, alpha=0.6, color='#0066cc', edgecolor='none')
x = np.linspace(rets.quantile(0.001), rets.quantile(0.999), 300)
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'r--', lw=2, label='Normal fit')
axes[0].set_title('Return Distribution')
axes[0].legend()

# Q-Q
stats.probplot(rets, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot vs Normal')

# Rolling vol (30-day)
monthly = df.groupby(df['date'].dt.to_period('M'))['daily_return'].std() * np.sqrt(252) * 100
monthly.plot(ax=axes[2], color='#ff8800')
axes[2].set_title('Annualised Vol Over Time (all tickers)')
axes[2].set_ylabel('%')

plt.tight_layout()
plt.show()

# Formal tests
ks_stat, ks_p = stats.kstest(rets.sample(5000, random_state=42), 'norm',
                               args=(rets.mean(), rets.std()))
jb_stat, jb_p = stats.jarque_bera(rets.dropna())
print(f'KS test:      statistic={ks_stat:.4f}  p-value={ks_p:.2e}')
print(f'Jarque-Bera:  statistic={jb_stat:.1f}  p-value={jb_p:.2e}')
print(f'Skewness:     {rets.skew():.4f}')
print(f'Excess Kurt:  {rets.kurtosis():.4f}')

## 5  Sector-Level Price Performance

In [ ]:
# Normalised cumulative returns from start (rebased to 100)
fig, ax = plt.subplots(figsize=(14, 6))
palette = {'Technology':'#0066cc','Finance':'#00994d','Healthcare':'#cc0000',
           'Energy':'#ff8800','Consumer':'#9900cc','Industrials':'#666666'}

for sector, grp in df.groupby('sector'):
    daily_sec = grp.groupby('date')['daily_return'].mean().fillna(0)
    cum_ret = (1 + daily_sec).cumprod() * 100
    ax.plot(cum_ret.index, cum_ret.values, label=sector,
            color=palette.get(sector, '#333333'), lw=1.5)

ax.axhline(100, color='grey', lw=0.8, ls='--')
ax.set_title('Cumulative Performance by Sector (rebased = 100)')
ax.set_ylabel('Index Value')
ax.legend()
plt.tight_layout()
plt.show()

## 6  Correlation Heatmap

In [ ]:
pivot  = df.pivot_table(index='date', columns='ticker', values='daily_return')
corr   = pivot.corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1,
            ax=ax, linewidths=0.3, annot=False, square=True)
ax.set_title('Ticker Return Correlation Matrix')
plt.tight_layout()
plt.show()

# Top 10 most correlated pairs
upper = corr.where(mask == False)
pairs = upper.stack().reset_index()
pairs.columns = ['ticker_a','ticker_b','correlation']
print('Top 10 most correlated pairs:')
print(pairs.nlargest(10,'correlation').to_string(index=False))

## 7  Volatility Clustering (ARCH Effects)

In [ ]:
ticker = 'AAPL'
sub = df[df['ticker'] == ticker].sort_values('date').dropna(subset=['daily_return'])

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(sub['date'], sub['daily_return'], lw=0.5, color='#0066cc')
axes[0].set_title(f'{ticker} Daily Returns')
axes[0].set_ylabel('Return')

axes[1].plot(sub['date'], sub['daily_return'].abs(), lw=0.5, color='#cc0000', alpha=0.7)
axes[1].set_title(f'{ticker} Absolute Returns (Volatility Proxy)')
axes[1].set_ylabel('|Return|')

plt.tight_layout()
plt.show()

# Ljung-Box test on squared returns
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(sub['daily_return'].dropna() ** 2, lags=[10, 20], return_df=True)
print('Ljung-Box on squared returns (ARCH effect test):')
print(lb)

## 8  Outlier Visualisation

In [ ]:
out_path = PROC_DIR / 'cy_outliers.parquet'
if out_path.exists():
    out_df = pd.read_parquet(out_path)
    out_df['date'] = pd.to_datetime(out_df['date'])
    print(f'Outlier rate: {out_df["is_outlier"].mean()*100:.2f}%')
    print(out_df.groupby('ticker')['is_outlier'].sum().sort_values(ascending=False).head(10))
    
    # Plot
    ticker = 'AAPL'
    sub  = out_df[out_df['ticker'] == ticker].sort_values('date')
    sub_o = sub[sub['is_outlier']]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(sub['date'], sub['close'], lw=0.8, color='#0066cc', label='Close')
    ax.scatter(sub_o['date'], sub_o['close'], color='red', s=20, zorder=5, label='Outlier')
    ax.set_title(f'{ticker} Price with Flagged Outliers')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Run src/outlier_detection.py first.')

## 9  Technical Indicator Analysis

In [ ]:
feat_path = PROC_DIR / 'cy_features.parquet'
if feat_path.exists():
    feat_df = pd.read_parquet(feat_path)
    feat_df['date'] = pd.to_datetime(feat_df['date'])

    ticker = 'AAPL'
    sub = feat_df[feat_df['ticker'] == ticker].sort_values('date').tail(500)

    fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

    axes[0].plot(sub['date'], sub['close'], color='#0066cc')
    axes[0].fill_between(sub['date'], sub['bb_lower'], sub['bb_upper'],
                         alpha=0.1, color='#0066cc')
    axes[0].set_title(f'{ticker} Price + Bollinger Bands')

    axes[1].plot(sub['date'], sub['rsi_14'], color='#ff8800')
    axes[1].axhline(70, ls='--', color='red', lw=1)
    axes[1].axhline(30, ls='--', color='green', lw=1)
    axes[1].set_title('RSI (14)')
    axes[1].set_ylim(0, 100)

    axes[2].plot(sub['date'], sub['macd_line'],   label='MACD',   color='#0066cc')
    axes[2].plot(sub['date'], sub['macd_signal'], label='Signal', color='#cc0000')
    axes[2].bar(sub['date'], sub['macd_hist'], alpha=0.4, color='#9900cc', label='Hist')
    axes[2].set_title('MACD')
    axes[2].legend(fontsize=8)

    axes[3].bar(sub['date'], sub['volume'], color='#00994d', alpha=0.7, width=0.8)
    axes[3].plot(sub['date'], sub['volume_ma20'], color='#cc0000', lw=1.2, label='MA20')
    axes[3].set_title('Volume vs 20-Day MA')
    axes[3].legend(fontsize=8)

    plt.suptitle(f'{ticker} — Full Technical Analysis', y=1.01, fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Run src/feature_engineering.py first.')

## 10  Key EDA Findings Summary

In [ ]:
sectors_ann = df.groupby('sector')['daily_return'].mean().mul(252*100).round(2)
sectors_vol  = df.groupby('sector')['daily_return'].std().mul(np.sqrt(252)*100).round(2)
summary = pd.DataFrame({'Ann Return %': sectors_ann, 'Ann Vol %': sectors_vol})
summary['Sharpe'] = (summary['Ann Return %'] - 5) / summary['Ann Vol %']
summary = summary.round(3)
print('=== Sector Risk-Return Summary ===')
print(summary.sort_values('Sharpe', ascending=False).to_string())